# Unbiased AI Decision — Data Audit
### Phase 1: Load datasets and find raw bias before touching any model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')
matplotlib.use('Agg')
os.makedirs('../reports/charts', exist_ok=True)
print('All imports successful')

All imports successful


In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
print('=== DATASET OVERVIEW ===')
print(f'Total employees: {len(df)}')
print(f'\nAttrition breakdown:')
print(df['Attrition'].value_counts())
print(f'\nOverall attrition rate: {(df["Attrition"]=="Yes").mean()*100:.1f}%')
print(f'Missing values: {df.isnull().sum().sum()}')
protected = ['Gender', 'Age', 'MaritalStatus']
print('\nProtected attributes:')
for col in protected:
    print(f'  {col}: {df[col].unique()}')

=== DATASET OVERVIEW ===
Total employees: 1470

Attrition breakdown:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Overall attrition rate: 16.1%
Missing values: 0

Protected attributes:
  Gender: ['Female' 'Male']
  Age: [41 49 37 33 27 32 59 30 38 36 35 29 31 34 28 22 53 24 21 42 44 46 39 43
 50 26 48 55 45 56 23 51 40 54 58 20 25 19 57 52 47 18 60]
  MaritalStatus: ['Single' 'Married' 'Divorced']


In [9]:
print('=== REPRESENTATION IMBALANCE ===')
gender_counts = df['Gender'].value_counts()
gender_pct = df['Gender'].value_counts(normalize=True) * 100
print('\nGender representation:')
for g in gender_counts.index:
    flag = ' << UNDERREPRESENTED' if gender_pct[g] < 35 else ''
    print(f'  {g}: {gender_counts[g]} ({gender_pct[g]:.1f}%){flag}')

df['AgeGroup'] = pd.cut(df['Age'], bins=[18,30,40,50,60], labels=['18-30','31-40','41-50','51-60'])
age_pct = df['AgeGroup'].value_counts(normalize=True) * 100
print('\nAge group representation:')
for a in age_pct.index:
    flag = ' << UNDERREPRESENTED' if age_pct[a] < 15 else ''
    print(f'  {a}: {age_pct[a]:.1f}%{flag}')

=== REPRESENTATION IMBALANCE ===

Gender representation:
  Male: 882 (60.0%)
  Female: 588 (40.0%)

Age group representation:
  31-40: 42.3%
  18-30: 25.9%
  41-50: 22.0%
  51-60: 9.8% << UNDERREPRESENTED


In [10]:
print('=== OUTCOME RATE GAPS ===')

gender_attrition = df.groupby('Gender')['Attrition'].apply(lambda x: (x=='Yes').mean() * 100)
print('\nAttrition rate by Gender:')
for g, rate in gender_attrition.items():
    print(f'  {g}: {rate:.1f}%')
gap = abs(gender_attrition.max() - gender_attrition.min())
flag = 'SIGNIFICANT GAP - BIAS DETECTED' if gap > 5 else 'Within acceptable range'
print(f'  Gap: {gap:.1f}% — {flag}')

age_attrition = df.groupby('AgeGroup')['Attrition'].apply(lambda x: (x=='Yes').mean() * 100)
print('\nAttrition rate by Age Group:')
for a, rate in age_attrition.items():
    print(f'  {a}: {rate:.1f}%')

marital_attrition = df.groupby('MaritalStatus')['Attrition'].apply(lambda x: (x=='Yes').mean() * 100)
print('\nAttrition rate by Marital Status:')
for m, rate in marital_attrition.items():
    print(f'  {m}: {rate:.1f}%')

=== OUTCOME RATE GAPS ===

Attrition rate by Gender:
  Female: 14.8%
  Male: 17.0%
  Gap: 2.2% — Within acceptable range

Attrition rate by Age Group:
  18-30: 25.4%
  31-40: 13.7%
  41-50: 10.6%
  51-60: 12.6%

Attrition rate by Marital Status:
  Divorced: 10.1%
  Married: 12.5%
  Single: 25.5%


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Data-Level Bias: Representation Imbalance', fontsize=14, fontweight='bold')

colors_gender = ['#378ADD' if p > 35 else '#E24B4A' for p in gender_pct.values]
axes[0].bar(gender_pct.index, gender_pct.values, color=colors_gender, edgecolor='white')
axes[0].set_title('Gender Representation (%)')
axes[0].set_ylabel('Percentage of Dataset')
axes[0].axhline(y=35, color='red', linestyle='--', alpha=0.7, label='Min threshold (35%)')
axes[0].legend(fontsize=9)
for i, (label, val) in enumerate(zip(gender_pct.index, gender_pct.values)):
    axes[0].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11)

age_colors = ['#E24B4A' if v < 15 else '#378ADD' for v in age_pct.values]
axes[1].bar(age_pct.index, age_pct.values, color=age_colors, edgecolor='white')
axes[1].set_title('Age Group Representation (%)')
axes[1].set_ylabel('Percentage of Dataset')
axes[1].axhline(y=15, color='red', linestyle='--', alpha=0.7, label='Min threshold (15%)')
axes[1].legend(fontsize=9)
for i, (label, val) in enumerate(zip(age_pct.index, age_pct.values)):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../reports/charts/01_representation_imbalance.png', dpi=150, bbox_inches='tight')
print('Chart 1 saved: 01_representation_imbalance.png')
plt.show()

Chart 1 saved: 01_representation_imbalance.png


In [12]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Data-Level Bias: Outcome Rate Gaps by Protected Group', fontsize=14, fontweight='bold')

axes[0].bar(gender_attrition.index, gender_attrition.values, color=['#378ADD','#E24B4A'], edgecolor='white')
axes[0].set_title('Attrition Rate by Gender')
axes[0].set_ylabel('Attrition Rate (%)')
for i, (label, val) in enumerate(zip(gender_attrition.index, gender_attrition.values)):
    axes[0].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

axes[1].bar(age_attrition.index, age_attrition.values,
            color=['#E24B4A','#BA7517','#378ADD','#1D9E75'], edgecolor='white')
axes[1].set_title('Attrition Rate by Age Group')
axes[1].set_ylabel('Attrition Rate (%)')
for i, (label, val) in enumerate(zip(age_attrition.index, age_attrition.values)):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=11)

axes[2].bar(marital_attrition.index, marital_attrition.values,
            color=['#534AB7','#1D9E75','#E24B4A'], edgecolor='white')
axes[2].set_title('Attrition Rate by Marital Status')
axes[2].set_ylabel('Attrition Rate (%)')
for i, (label, val) in enumerate(zip(marital_attrition.index, marital_attrition.values)):
    axes[2].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../reports/charts/02_outcome_rate_gaps.png', dpi=150, bbox_inches='tight')
print('Chart 2 saved: 02_outcome_rate_gaps.png')
plt.show()

Chart 2 saved: 02_outcome_rate_gaps.png


In [13]:
print('=== INTERSECTIONAL ANALYSIS: Gender x Age Group ===')
intersect = df.groupby(['Gender','AgeGroup'])['Attrition'].apply(
    lambda x: (x=='Yes').mean() * 100
).unstack()
print(intersect.round(1))

fig, ax = plt.subplots(figsize=(10, 5))
intersect.T.plot(kind='bar', ax=ax, color=['#378ADD','#E24B4A'], edgecolor='white')
ax.set_title('Intersectional Bias: Attrition Rate by Gender x Age Group',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Attrition Rate (%)')
ax.legend(title='Gender')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../reports/charts/03_intersectional_analysis.png', dpi=150, bbox_inches='tight')
print('Chart 3 saved: 03_intersectional_analysis.png')
plt.show()

=== INTERSECTIONAL ANALYSIS: Gender x Age Group ===
AgeGroup  18-30  31-40  41-50  51-60
Gender                              
Female     28.2   11.2   10.1    7.8
Male       23.7   15.4   10.9   16.5
Chart 3 saved: 03_intersectional_analysis.png


In [14]:
df.to_csv('../data/ibm_hr_with_agegroup.csv', index=False)
print('Clean dataset saved: data/ibm_hr_with_agegroup.csv')
print('\n=== DATA AUDIT COMPLETE ===')
print('3 charts saved in reports/charts/')
print('Next: open model_training.ipynb')

Clean dataset saved: data/ibm_hr_with_agegroup.csv

=== DATA AUDIT COMPLETE ===
3 charts saved in reports/charts/
Next: open model_training.ipynb
